# 1. Case Definition && Business Understanding

## 1.1 Problem Definition

Case Scenario: You are working as a Senior Data Scientist at a Klavi a firm specialized in credit risk. A Fintech Client wants to review their credit granting policy and needs to define:

 Which variables are most relevant for predicting default

 Which variables are good candidates for a practical credit policy (rules + score)

 How to turn these variables into an objective and justifiable proposal for the business team


 Challenge: You are given one week to analyze the data files provided by the client. You must deliver a recommendation on how to proceed using Klavi’s variables. Using a visual approach, show how to analyze the data and extract meaningful business insights.

Please detail:
 The main steps you would take to explore, clean, and organize the provided dataset.
 The key factors or criteria you would consider essential to ensure the insights are relevant, accurate, and representative.
 How you would validate both the analytical approach and the insights generated.

Please fell free to contact Andrea Filgueiras at andrea.filgueiras@klavi.ai if you have any questions

## 1.2 Objective of this study

The objetive of this notebook is provide evidences to give an anwser to the questions below:

1. Which variables are most relevant for predicting default?

2. Which variables are good candidates for a pratical credit policy (rules + score)?

3. How to turn these variables into an objective and justifiable proposal for the business team?

Challenge: You are given one week to analyze the data files provided by the client. You must deliver a recommendation on how to proceed using Klavi’s variables. Using a visual approach, show how to analyze the data and extract meaningful business insights.

In other words, the objective of this analysis is to identify the most relevant variables for predicting credit default and to propose a practical and explainable credit policy. The final solution must support business decision-making by translating data insights into actionable rules and/or a scoring system.


## 1.3 Business Understanding

At this stage, the definition of "default" is assumed to be provided in the dataset as a binary variable.

However, it is critical to validate:
- The time horizon of default (e.g., 30, 60, 90 days past due)
- Whether the target includes post-decision information (data leakage risk)
- The business meaning of default (financial loss vs delay)

This validation will be performed during the data exploration phase.


The proposed solution must consider:
- Interpretability: variables must be explainable to business stakeholders
- Simplicity: rules should be implementable in production systems
- Stability: variables must be robust over time
- Regulatory compliance: avoid non-transparent or biased variables

The success of the solution will be evaluated based on:
- Predictive power (e.g., AUC, KS, separation between good/bad)
- Business interpretability
- Ease of implementation
- Alignment with business objectives
- Robustness and stability

# 2. Import Libs && DataBase

In [1]:
import pandas as pd
import numpy as np

In [2]:
base_profile = pd.read_parquet("./profile_case_credit.parquet")

base_transactions = pd.read_parquet("./transactions_case_credit.parquet")

# 3. EDA - Explore Data Analysis - Data Undertanding

## 3.1 Data Undertanding 

### 3.1.1 DataBase transactions

In [5]:
base_transactions.columns

Index(['uuid', 'REFERENCIA', 'score_klavi_1', 'score_klavi_2', 'classe_social',
       'status_empregaticio', 'regiao', 'Renda', 'risco_aposta',
       'Capacidade Financeira'],
      dtype='str')

In [17]:
base_transactions.info()

<class 'pandas.DataFrame'>
Index: 371596 entries, 0 to 1198093
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   uuid                   371596 non-null  str    
 1   REFERENCIA             371596 non-null  str    
 2   score_klavi_1          32277 non-null   float64
 3   score_klavi_2          331573 non-null  float64
 4   classe_social          371596 non-null  str    
 5   status_empregaticio    371596 non-null  str    
 6   regiao                 371013 non-null  str    
 7   Renda                  32277 non-null   float64
 8   risco_aposta           32277 non-null   float64
 9   Capacidade Financeira  325286 non-null  float64
dtypes: float64(5), str(5)
memory usage: 52.7 MB


In [8]:
base_transactions.head(5)

,uuid,REFERENCIA,score_klavi_1,score_klavi_2,classe_social,status_empregaticio,regiao,Renda,risco_aposta,Capacidade Financeira
0,7b0c9b36fe0f730f7c0d081534ff9a2b,2024-06-12,58.07,512.0,D/E,Desconhecido,Sudeste,1052.65,0.0,839.64
3,513b46119c7d33fc324f3c561905c159,2025-02-06,NaN,422.0,C,Desconhecido,Sudeste,NaN,NaN,2213.37
7,0d7fed4a5b92cbae24e2b80e49fb0a4e,2025-06-13,NaN,607.0,C,Desconhecido,Sudeste,NaN,NaN,587.04
8,c158eeca94b81f5f63cee1496dbea081,2024-07-30,NaN,NaN,C,Autônomo,Sudeste,NaN,NaN,1961.59
9,97ebff4ac03d546fcb03040d1866bea0,2024-11-04,NaN,533.0,B,Desconhecido,Sudeste,NaN,NaN,3585.16


In [9]:
base_transactions.describe()

,score_klavi_1,score_klavi_2,Renda,risco_aposta,Capacidade Financeira
count,32277.000000,331573.000000,32277.000000,32277.000000,3.252860e+05
mean,-72475.672167,520.963981,2850.452457,1.412585,1.925911e+03
std,259379.346501,60.696365,3600.016877,2.739815,5.038808e+03
min,-999999.000000,345.000000,317.280000,0.000000,-1.088145e+06
25%,40.340000,481.000000,1321.570000,0.000000,8.038300e+02
50%,57.250000,524.000000,2066.940000,0.000000,1.470810e+03
75%,68.190000,566.000000,3057.770000,1.000000,2.361590e+03
max,100.000000,663.000000,129296.190000,10.000000,1.207959e+05


In [14]:
# na base transactions, contar os valores distintos da combinacao de uuid + referencia
base_transactions.groupby(['uuid', 'REFERENCIA']).size().reset_index(name='counts').describe()

,counts
count,371596.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


### 3.1.2 DataBase profile

Which columns are in this dataframe?

In [4]:
base_profile.columns

Index(['uuid', 'DATA', 'SAFRA', 'OVER30M2', 'STATUS'], dtype='str')

What information do we have about the data types of these columns and the dataset size?

In [18]:
base_profile.info()

<class 'pandas.DataFrame'>
RangeIndex: 220618 entries, 0 to 220617
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   uuid      220618 non-null  str    
 1   DATA      220618 non-null  str    
 2   SAFRA     220618 non-null  float64
 3   OVER30M2  24148 non-null   float64
 4   STATUS    220618 non-null  str    
dtypes: float64(2), str(3)
memory usage: 19.1 MB


In [34]:
24148/220618 

0.10945616404826441

The date column is currently in string format, so it needs to be converted to a datetime type. This will allow us to perform date-related operations and analyses.

The safra column can also be treated as a datetime variable.

The OVER30M2 column contains a large number of missing values. Only 24.148 rows have valid values, while 196.470 are missing. This means that only about 10% of the dataset contains values for the potential target variable.

Lets see the first ten lines of this dataset

In [28]:
base_profile.head(10)

,uuid,DATA,SAFRA,OVER30M2,STATUS
0,d946dabe8351c29c3ca9db3c7005341b,2024-08-24,202408.0,0.0,Contratado
1,b9aa3d3552e2e21f92d34df2540de5c3,2024-09-02,202409.0,1.0,Contratado
2,ac5371c706da6f7a57c9b9593a8a5d04,2024-11-25,202411.0,0.0,Contratado
3,d44880bed90eaacb2fba3feead83ca16,2024-09-09,202409.0,0.0,Contratado
4,8ec0eefeda7ae58d5a98a2d59aca4bbb,2024-06-18,202406.0,0.0,Contratado
5,f678146b3eca8e6712135fddc35fb3ba,2024-09-07,202409.0,0.0,Contratado
6,9dea08ff1d666272f3c60e56e3398876,2024-11-19,202411.0,0.0,Contratado
7,862fdc1869d84d8adbae80677c99b7b3,2025-05-28,202505.0,0.0,Contratado
8,010b18ce36ae05c53de14917e90191b3,2025-05-08,202505.0,0.0,Contratado
9,bbb827213dcec48e8725392cf0423273,2025-06-20,202506.0,0.0,Contratado


At first glance, there is one numeric variable (safra, although it refers to a date), one date variable (in the format), and two categorical variables (over30m2, apparently the target, and status)

In [3]:
base_profile.describe()

,SAFRA,OVER30M2
count,220618.000000,24148.000000
mean,202450.283331,0.103031
std,46.821529,0.304006
min,202406.000000,0.000000
25%,202409.000000,0.000000
50%,202412.000000,0.000000
75%,202503.000000,0.000000
max,202506.000000,1.000000


In [31]:
# convertendo a coluna DATA para datetime
base_profile['DATA'] = pd.to_datetime(base_profile['DATA'], errors='coerce')

In [36]:
base_profile['DATA'].min(), base_profile['DATA'].max()

(Timestamp('2024-06-01 00:00:00'), Timestamp('2025-06-30 00:00:00'))

In [37]:
# convertendo a coluna Safra para datetime
base_profile['SAFRA'] = pd.to_datetime(base_profile['SAFRA'], errors='coerce')

In [38]:
base_profile['SAFRA'].min(), base_profile['SAFRA'].max()

(Timestamp('1970-01-01 00:00:00.000202406'),
 Timestamp('1970-01-01 00:00:00.000202506'))

In [29]:
base_profile['STATUS'].value_counts()

STATUS
Reprovado     119247
Aprovado       77223
Contratado     24148
Name: count, dtype: int64

In [22]:
# na base profile, contar os valores distintos da combinacao de uuid + safra
base_profile.groupby(['uuid', 'SAFRA']).size().reset_index(name='counts').describe()

,SAFRA,counts
count,209838.000000,209838.000000
mean,202450.243297,1.051373
std,46.812752,0.285252
min,202406.000000,1.000000
25%,202409.000000,1.000000
50%,202412.000000,1.000000
75%,202503.000000,1.000000
max,202506.000000,8.000000


In [21]:
# na base profile, contar os valores distintos da combinacao de uuid + data
base_profile.groupby(['uuid', 'DATA']).size().reset_index(name='counts').describe()

,counts
count,220618.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [19]:
base_profile['OVER30M2'].value_counts() 

OVER30M2
0.0    21660
1.0     2488
Name: count, dtype: int64

In [23]:
base_profile['OVER30M2'].value_counts(normalize=True) 

OVER30M2
0.0    0.896969
1.0    0.103031
Name: proportion, dtype: float64

## 3.2 Data Quality Assessment

## 3.3 Conclusion